# Minería de Datos · Sesión 9: Clustering
## Segmentación de Criminalidad y Parque Automotor: K-Means y Jerárquico Aglomerativo
**Proyecto:** Análisis del Hurto de Vehículos y Capacidad Policial en Colombia  
**Docente:** Esteban Ernesto Morales Castro · UNIMINUTO Sede Ibagué · 2026-2  
**Instrumento:** Ejercicios en Clase / Segmentación y Campañas (Unidad 2 - PEA)

---

### Resumen de la Misión:
En las sesiones anteriores trabajamos con aprendizaje supervisado guiados por la columna de hurtos.  
En esta sesión se apagan las etiquetas de respuesta para descubrir la **estructura latente y los segmentos naturales** del país mediante minería de datos no supervisada:
1. **Diferenciación Conceptual:** Aprendizaje supervisado vs no supervisado; clustering sin contaminación de la variable objetivo (la tasa de hurto se usa para perfilar a posteriori).
2. **K-Means Paso a Paso:** Sembrado inteligente (`k-means++`), asignación por distancia euclídea y recalculo iterativo de centroides hacia la mínima inercia.
3. **Selección Objetiva de k:** Método del Codo (Inercia) y Coeficiente de Silueta evaluados para $k \in [2..8]$.
4. **Perfilamiento con Medias Reales:** Análisis descriptivo de cada segmento con variables en escala natural y cruce con efectividad policial.
5. **Retrato Robot y Bautizo Estratégico:** Nombres comerciales y criminológicos para cada clúster, con una acción de seguridad pública específica por grupo.
6. **Clustering Jerárquico Aglomerativo:** Construcción de dendrograma con método `ward` sobre muestra de 60 municipios para validar el corte de ramas.
7. **Exportación Enriquecida:** Guardado de `dataset_con_clusters.csv` para potenciar los modelos del Avance 2.


## 1. Importación de Librerías y Módulo de Clustering


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn y scipy
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram

# Carga modular del proyecto
sys.path.append('../src')
from clustering import SegmentacionClustering

print("✅ Entorno de la Sesión 9 inicializado correctamente.")


✅ Entorno de la Sesión 9 inicializado correctamente.


## 2. Carga del Dataset y Selección de Variables Operativas
Seleccionamos las variables estructurales (`AÑO`, `VEHICULOS_REGISTRADOS`, `VEHICULOS_RECUPERADOS`).  
**Regla Metodológica Clave:** No incluimos `VEHICULOS_HURTADOS` en la matriz $X$ para evitar fuga no supervisada. La criminalidad se analiza al perfilar.


In [2]:
seg = SegmentacionClustering('../dataset/dataset_integrado.csv')
df = seg.cargar()
print(f"Total registros cargados: {len(df):,}")
X_scaled = seg.preparar_datos()
print(f"Matriz estandarizada X_scaled: {X_scaled.shape[0]:,} filas × {X_scaled.shape[1]} variables")


Total registros cargados: 54,428
Matriz estandarizada X_scaled: 54,428 filas × 3 variables


## 3. Selección Objetiva de k: Método del Codo y Coeficiente de Silueta
Evaluamos $k \in [2..8]$ calculando la inercia (compacidad intraclúster) y la silueta promedio (separación interclúster):


In [3]:
df_eval = seg.evaluar_codo_y_silueta(k_rango=range(2, 9))
print(df_eval)


   k   inercia  silueta
0  2  103591.4   0.7550
1  3   63857.9   0.5450
2  4   42790.1   0.5590
3  5   34948.6   0.4406
4  6   28236.6   0.4738
5  7   23818.8   0.4751
6  8   19748.1   0.4876


## 4. Ajuste del Modelo K-Means Definitivo (k=3)
Elegimos $k=3$ porque representa el punto de inflexión del codo (la inercia cae fuertemente de $103.591$ a $63.857$), mantiene una silueta de $0,545$ y responde a la capacidad operativa de gestionar 3 estrategias de seguridad diferenciadas.


In [4]:
conteo = seg.ajustar_kmeans_final(k=3, n_init=10, random_state=42)
print("Distribución de observaciones por clúster:")
print(conteo)


Distribución de observaciones por clúster:
CLUSTER_KMEANS
0    15471
1    36633
2     2324
Name: count, dtype: int64


## 5. Perfilamiento Oficial con Medias Reales (groupby)
Calculamos las medias en sus unidades originales (años, parque automotor, hurtos y recuperaciones reales), complementadas con la tasa de criminalidad activa y el ratio de efectividad policial:


In [5]:
perfiles = seg.perfilar_clusters()
print(perfiles)


                    AÑO  ...  ratio_recuperacion_pct
CLUSTER_KMEANS           ...                        
0               1974.93  ...                     0.0
1               2013.70  ...                    43.7
2               2017.18  ...                    42.8

[3 rows x 8 columns]


## 6. Bautizo Estratégico de Segmentos y Campañas Operativas
Asignamos nombres estratégicos ('Retrato Robot') y una acción concreta de seguridad pública para cada segmento:


In [6]:
estrategias = seg.asignar_nombres_estrategicos()
for idx, r in estrategias.iterrows():
    print(f"=== CLÚSTER {idx}: {r['nombre_estrategico']} ===")
    print(f"• Volumen nacional: {r['n_observaciones']:,} registros ({r['pct_total_pais']}%)")
    print(f"• Año medio: {r['AÑO']:.1f} | Parque: {r['VEHICULOS_REGISTRADOS']:.1f} veh | Hurtos: {r['VEHICULOS_HURTADOS']:.1f}")
    print(f"• Tasa de hurto activo: {r['tasa_hurto_activa_pct']}% | Efectividad policial: {r['ratio_recuperacion_pct']}%")
    print(f"• {r['estrategia_policial']}\n")


=== CLÚSTER 0: S1 · Archivo Histórico Premetropolitano (Inmune) ===
• Volumen nacional: 15,471.0 registros (28.4%)
• Año medio: 1974.9 | Parque: 107.5 veh | Hurtos: 0.0
• Tasa de hurto activo: 0.0% | Efectividad policial: 0.0%
• Acción: Depuración y consolidación documental de registros vehiculares antiguos en el RUNT. Cero costo operativo policial en campo; solo auditoría de bases de datos.

=== CLÚSTER 1: S2 · Cabeceras Municipales con Delincuencia Contenida ===
• Volumen nacional: 36,633.0 registros (67.3%)
• Año medio: 2013.7 | Parque: 174.8 veh | Hurtos: 5.6
• Tasa de hurto activo: 60.8% | Efectividad policial: 43.7%
• Acción: Fortalecimiento de cuadrantes preventivos, patrullaje en zonas comerciales y puestos de control relámpago contra el halado de motocicletas en fines de semana.

=== CLÚSTER 2: S3 · Grandes Focos Metropolitanos de Hurto Masivo ===
• Volumen nacional: 2,324.0 registros (4.3%)
• Año medio: 2017.2 | Parque: 4634.5 veh | Hurtos: 78.3
• Tasa de hurto activo: 79.6% 

## 7. Clustering Jerárquico Aglomerativo (Ward sobre Muestra de 60 Casos)
Construimos el dendrograma jerárquico para verificar visualmente la jerarquía natural de fusiones:


In [7]:
res_jer = seg.ejecutar_clustering_jerarquico(n_muestra=60, random_state=42)
for k, v in res_jer.items():
    print(f"• {k:<34}: {v}")


• n_muestra                         : 60
• metodo_enlace                     : ward
• primeras_5_fusiones_distancias    : [0.0, 0.0, 0.0, 0.01, 0.04]
• ultimas_3_fusiones_distancias     : [6.64, 9.62, 12.17]
• distancia_maxima_raiz             : 12.17
• altura_corte_recomendada          : 6.3
• num_clusters_corte                : 3


## 8. Galería de Visualizaciones de Clustering
Generamos y guardamos las tres figuras oficiales en `analisis/figuras/`:
1. `fig_sesion9_metodo_codo_silueta.png`
2. `fig_sesion9_perfilamiento_clusters.png`
3. `fig_sesion9_dendrograma_muestra60.png`


In [8]:
figs = seg.generar_graficos('../analisis/figuras')
for f in figs:
    print("Figura oficial generada:", f)


✅ Figuras de la Sesión 9 guardadas en: ../analisis/figuras
Figura oficial generada: ../analisis/figuras/fig_sesion9_metodo_codo_silueta.png
Figura oficial generada: ../analisis/figuras/fig_sesion9_perfilamiento_clusters.png
Figura oficial generada: ../analisis/figuras/fig_sesion9_dendrograma_muestra60.png


### Visualización 1: Método del Codo y Silueta
![Codo y Silueta](figuras/fig_sesion9_metodo_codo_silueta.png)

### Visualización 2: Perfilamiento de Segmentos
![Perfiles de Clústeres](figuras/fig_sesion9_perfilamiento_clusters.png)

### Visualización 3: Dendrograma Jerárquico
![Dendrograma](figuras/fig_sesion9_dendrograma_muestra60.png)


## 9. Exportación del Dataset Enriquecido (Alimento para Avance 2)
Guardamos el dataset integrado con las columnas `CLUSTER_KMEANS` y `SEGMENTO_ESTRATEGICO`:


In [9]:
ruta_csv = seg.exportar_dataset_con_clusters('../dataset/dataset_con_clusters.csv')
print("Dataset enriquecido disponible en:", ruta_csv)


✅ Dataset enriquecido guardado exitosamente en: ../dataset/dataset_con_clusters.csv
Dataset enriquecido disponible en: ../dataset/dataset_con_clusters.csv
